In [2]:
import os
from groq import Groq
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

print("Groq client initialized")
print("Day 10 - LLM + Prompt Engineering")

# Quick Test
response = client.chat.completions.create(
    model = "llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Say 'RAG pipeline is ready' and nothing else."}
    ]
)

print(f"\nTest response: {response.choices[0].message.content}")
print(f"Model used: {response.model}")
print(f"Tokens used: {response.usage.total_tokens}")

Groq client initialized
Day 10 - LLM + Prompt Engineering

Test response: RAG pipeline is ready
Model used: llama-3.1-8b-instant
Tokens used: 53


In [8]:
print("=== Chat Message Structure ===\n")

# Every conversation has 3 roles:
# system -> sets the LLM's personality and rules
# user   -> the human's message
# assistant -> the LLM's response

context = """RAG combines retrieval and generation. 
It first retrieves relevant documents then generates an answer based on them.
Hybrid search uses both BM25 and vector search merged with RRF."""

question = "How does hybrid search work in RAG?"

# Combine into single user message string
user_message = f"Context:\n{context}\n\nQuestion:\n{question}"

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": "You are an AI assistant for an Enterprise RAG pipeline. Answer questions based only on the provided context. If the answer is not in the context say 'I cannot find this in the provided documents.' Never make up information. Be concise and precise."
        },
        {
            "role": "user",
            "content": user_message
        }
    ],
    temperature=0.1,
    max_tokens=200
)

answer = response.choices[0].message.content
print(f"System prompt: sets behaviour and rules")
print(f"User message: contains context + question")
print(f"\nLLM Answers:\n{answer}")
print(f"\nTokens - prompt: {response.usage.prompt_tokens}")
print(f"Tokens - Response: {response.usage.completion_tokens}")
print(f"Token - Total: {response.usage.total_tokens}")

=== Chat Message Structure ===

System prompt: sets behaviour and rules
User message: contains context + question

LLM Answers:
Hybrid search in RAG combines both BM25 and vector search with RRF (Re-Ranker). It merges the results from these two search methods to provide a more comprehensive and accurate search outcome.

Tokens - prompt: 134
Tokens - Response: 41
Token - Total: 175


In [ ]:
print("=== Prompt Engineering Patterns ===\n")

def ask_llm(system: str, user: str, temperature: float=0.1)-> str:
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        temperature=temperature,
        max_tokens=300
    )
    return response.choices[0].message.content

# Pattern 1 - Zero Shot
print("=== Pattern 1: Zero-shot ===")
zero_shot = ask_llm(
    system="You are a helpful AI assistant.",
    user="Classify this text as TECHNICAL, LEGAL, or FINANCIAL:\n'The quarterly revenue exceeded projections by 23 percent due to strong product sales.'"
)
print(f"Zero-shot: {zero_shot}\n")

# Pattern 2 - Few-shot
print("=== Pattern 2: Few-Shot ===")
few_shot = ask_llm(
    system="you are a document classifier.",
    user="""Classify documents as TECHNICAL, LEGAL, or FINANCIAL.
    
Examples:
Text: "The API endpoint returns a 404 error when the token expires"
Category: TECHNICAL

Text: "The contract clause limits liability to direct damages only"
Category: LEGAL

Text: "Operating expenses increased by 12 percent year over year"
Category: FINANCIAL

Now classify this:
Text: "This neural network achieved 94 percent accuracy on the test set"
Category:"""
)

print(f"Few-shot: {few_shot}\n")

# Pattern 3 - Chain of Thought
print("=== Pattern 3: Chain of Thought ===")
cot = ask_llm(
    system="You are an AI assistant. Think step by step before answering.",
    user="""A RAG pipeline processes 1000 documents per day.
Each document has 120 chunks on average.
Each chunk costs 0.0001 dollars to embed.
The system runs 365 days per year.
What is the annual embedding cost?
Think step by step."""
)
print(f"Chain of Thoughts:\n{cot}\n")

=== Prompt Engineering Patterns ===

=== Pattern 1: Zero-shot ===
Zero-shot: I would classify this text as FINANCIAL. It discusses revenue and sales, which are key financial metrics.

=== Pattern 2: Few-Shot ===
Few-shot: Category: TECHNICAL

=== Pattern 3: Chain of Thought ===
Chain of Thoughts:
To find the annual embedding cost, we need to break down the problem into smaller steps.

Step 1: Calculate the daily embedding cost.
- Number of documents per day: 1000
- Number of chunks per document: 120
- Cost per chunk: 0.0001 dollars
- Total chunks per day: 1000 * 120 = 120,000
- Daily embedding cost: 120,000 * 0.0001 = 12 dollars

Step 2: Calculate the annual embedding cost.
- Number of days per year: 365
- Daily embedding cost: 12 dollars
- Annual embedding cost: 12 * 365 = 4380 dollars

Therefore, the annual embedding cost is 4380 dollars.



In [24]:
from typing import List, Dict, Optional
from dataclasses import dataclass

@dataclass
class RAGPromptTemplate:
    """
    Manages prompts for the Enterprise RAG Pipeline.
    Every LLM call goes through the class.
    """
    system_prompt: str
    max_context_length: int = 2000
    temperature: float = 0.1
    max_tokens: int = 500

    def build_rag_prompt(
            self,
            query: str,
            retrieved_chunks: List[Dict],
            conversation_history: Optional[List[Dict]] = None
    ) -> List[Dict]:
        """ 
        Build a complete message list for RAG response generation.

        Args:
            query: user's question
            retrieved_chunks: list of {text, score, source} dicts
            converation_history: previous messages for multi-turn chat

        Returns:
            List of messages ready for LLM
        """
        # Build context from retrieved chunks
        context_parts = []
        for i, chunk in enumerate(retrieved_chunks):
            context_parts.append(
                f"[Source {i+1}: {chunk['source']} | Score: {chunk['score']:.3f}]\n"
                f"{chunk['text']}"
            )
        context = "\n\n".join(context_parts)

        # Truncate if too long
        if len(context) > self.max_context_length:
            context = context[:self.max_context_length] + "...[truncated]"

        # Build user message
        user_message = f"""Here are relevant documents retrieved for your question:

{context}

Based on the documents above, please answer this question: {query}

Remember to cite which source you used and only use information from the documents provided."""
        
        # Build message list
        messages = [{"role": "user", "content": user_message}]
        return messages
    
    def generate(
            self, 
            query: str,
            retrieved_chunks: List[Dict],
            conversation_history: Optional[List[Dict]] = None
    ) -> Dict:
        """Generate a RAG response and return full result with metadata"""
        messages = self.build_rag_prompt(query, retrieved_chunks, conversation_history)

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            temperature= self.temperature,
            max_tokens=self.max_tokens
        )

        return {
            "answer": response.choices[0].message.content,
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens,
            "model": response.model,
            "num_chunks_used": len(retrieved_chunks)
        }
    
# Initialize templates
rag_template = RAGPromptTemplate(
    system_prompt="""You are an Enterprise RAG assistant helping users
query internal documents. You:
- Answer only from provided context
- Always cite your source
- Say clearly when information is not available
- Are concise and professional
- Never hallucinate or make up information""",
    temperature=0.1,
    max_tokens=400
)

# Simulate retrieved chunks from your pipeline
retrieved_chunks = [
    {
        "text": "Hybrid search combines BM25 keyword search with vector semantic search. Results are merged using Reciprocal Rank Fusion (RRF) which assigns scores based on rank position rather than raw scores.",
        "score": 0.89,
        "source": "rag_architecture.pdf"
    },
    {
        "text": "Re-ranking uses a cross-encoder model to re-score the top retrieved chunks. Unlike bi-encoders used in retrieval, cross-encoders process query and document together for higher precision.",
        "score": 0.81,
        "source": "retrieval_methods.pdf"
    },
    {
        "text": "RAGAs evaluates pipelines using faithfulness, answer relevancy, context recall and context precision metrics. Faithfulness measures if thr answer is grounded in retrieved context.",
        "score": 0.76,
        "source": "evaluation_guide.pdf"
    }
]

# Test single query
print("=== Single Query ===\n")
result = rag_template.generate(
    query="How does hybrid search improve retrieval quality?",
    retrieved_chunks=retrieved_chunks
)

print(f"Answer:\n{result['answer']}")
print(f"\nTokens used: {result['total_tokens']}")
print(f"Chunks used: {result['num_chunks_used']}")

=== Single Query ===

Answer:
According to Source 1: rag_architecture.pdf, hybrid search improves retrieval quality by combining BM25 keyword search with vector semantic search. The results are then merged using Reciprocal Rank Fusion (RRF), which assigns scores based on rank position rather than raw scores. This combination of search methods and score fusion technique likely leads to improved retrieval quality.

Tokens used: 307
Chunks used: 3


In [25]:
print("=== Multi-turn Conversation ===\n")

# First turn 
result1 = rag_template.generate(
    query="What is RRF and how does it work?",
    retrieved_chunks=retrieved_chunks
)
print(f"Q1: What is RRF and how does it work?")
print(f"A1: {result1['answer']}\n")

# Build conversation history from first turn
history = [
    {
        "role": "user",
        "content": "What is RRF and how does it work?"
    },
    {
        "role": "assistant",
        "content": result["answer"]
    }
]

# Second turn - follow up question
result2 = rag_template.generate(
    query="What about re-ranking - how is it different from RRF?",
    retrieved_chunks=retrieved_chunks,
    conversation_history=history
)
print(f"Q2: What about re-ranking - how is it different from RRF?")
print(f"A2: {result2['answer']}\n")

# Test "I don't know case"
print("=== Testing Unknown Query ===\n")
result3 = rag_template.generate(
    query = "What is the pricing model for this RAG system?",
    retrieved_chunks=retrieved_chunks
)
print(f"Q3: What is the pricing model for this RAG system?")
print(f"A3: {result3 ['answer']}")

=== Multi-turn Conversation ===

Q1: What is RRF and how does it work?
A1: According to [Source 1: rag_architecture.pdf | Score: 0.890], RRF stands for Reciprocal Rank Fusion. It is a method used to merge the results of BM25 keyword search and vector semantic search.

RRF assigns scores based on rank position rather than raw scores. This means that the scores of the merged results are determined by the position of the results in the ranking, rather than their original scores from the individual search methods.

No further information about the specifics of how RRF works is provided in the given documents.

Q2: What about re-ranking - how is it different from RRF?
A2: According to Source 2: retrieval_methods.pdf, re-ranking uses a cross-encoder model to re-score the top retrieved chunks. This is different from Reciprocal Rank Fusion (RRF) used in Hybrid search (Source 1: rag_architecture.pdf), which assigns scores based on rank position rather than raw scores.

=== Testing Unknown Query